## Timestamp Granularity

**Information Need:** Determine the effective recording granularity and timezone of timestamps in an event log, assess whether they vary across events, identify temporal components that remain constant throughout the log, and, where required, normalize them to a selected temporal granularity..

**Motivation:** Timestamp recording granularity and timezone may be unknown or vary across events or activity types. Insufficient or heterogeneous timestamp granularity/timezone can result in events sharing identical timestamps, making their ordering ambiguous, introducing apparent concurrency, and potentially affecting subsequent temporal analyses. Moreover, temporal components that are constant throughout the log, such as the year or month, reveal the effective temporal scope of the recorded data and may be relevant when interpreting temporal patterns. Normalizing timestamps to a common granularity can provide a consistent temporal representation at the resolution required for subsequent analysis.

**Precondition:** A target granularity is specified if normalization is performed.

**Approach:** Characterize event timestamps according to their observed temporal resolution and timezone, and assess the variation of timestamp components across the event log, identifying both the effective granularity of individual timestamps and higher-level temporal components that remain constant. If a target granularity is specified, transform timestamps by removing temporal information below that level.

**Output:** The observed timestamp granularity levels and their distribution across events, together with the timezone, and any temporal components that remain constant throughout the event log and their corresponding values; if normalization is performed, an event log whose timestamps are represented at the selected target granularity.

*Note that at the time of implementing this notebook, pm4py.read_xes/write_xes convert to UTC and silently drop original timezone information. Consider csv format for exporting event logs with relevant timezone information*

In [ ]:
import pandas as pd
import pm4py
import re
import pytz


# --- Configuration -----------------------------------------------------------
LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

## Pattern execution

In [ ]:
TIMESTAMP_LEVELS = ['year', 'month', 'day', 'hour', 'minute', 'second', 'millisecond', 'sub-millisecond']

# Automatically analyze every column that pandas recognizes as datetime-like.
# This includes both timezone-naive datetime64 columns and timezone-aware datetime64 columns.
TIMESTAMP_COLUMNS = [
    c for c in event_log.columns
    if pd.api.types.is_datetime64_any_dtype(event_log[c])
]

print(f"Detected {len(TIMESTAMP_COLUMNS)} timestamp column(s): {TIMESTAMP_COLUMNS}")

### 1. Identify the level of granularity for each event

In [ ]:
def classify_granularity(ts):
    if pd.isna(ts):
        return None
    if ts.nanosecond != 0 or ts.microsecond % 1000 != 0:
        return 'sub-millisecond'
    if ts.microsecond != 0:
        return 'millisecond'
    if ts.second != 0:
        return 'second'
    if ts.minute != 0:
        return 'minute'
    if ts.hour != 0:
        return 'hour'
    return 'day'

granularity_distributions = {}

for column in TIMESTAMP_COLUMNS:
    granularity = event_log[column].apply(classify_granularity)
    counts = granularity.value_counts().reindex(TIMESTAMP_LEVELS, fill_value=0)
    percentages = (counts / len(granularity) * 100).round(2)

    distribution = pd.DataFrame({
        'Timestamp column': column,
        'Granularity': TIMESTAMP_LEVELS,
        'Events': counts.values,
        'Percentage': percentages.values
    })

    granularity_distributions[column] = distribution

if granularity_distributions:
    granularity_summary = pd.concat(granularity_distributions.values(), ignore_index=True)
    display(granularity_summary)
else:
    print("No datetime-like timestamp columns detected.")


### 2. Identify if any timestamp elements are constant

Check whether any timestamp components are constant across the whole log (e.g. all events on the same year, or same year and month), starting from the coarsest component and stopping at the first one that varies.

In [ ]:
def get_components(ts):
    return (
        ts.year, ts.month, ts.day, ts.hour, ts.minute, ts.second,
        ts.microsecond // 1000,
        (ts.microsecond % 1000) * 1000 + ts.nanosecond,
    )

constant_summaries = {}

for column in TIMESTAMP_COLUMNS:
    timestamps = event_log[column].dropna()

    print(f"\n--- {column} ---")

    if timestamps.empty:
        print("No non-missing timestamps available.")
        continue

    components_df = pd.DataFrame(
        timestamps.apply(get_components).tolist(),
        columns=TIMESTAMP_LEVELS,
        index=timestamps.index,
    )

    unique_counts = components_df.nunique()
    is_constant = unique_counts == 1

    constant_summary = pd.DataFrame({
        'Timestamp column': column,
        'Level': TIMESTAMP_LEVELS,
        'Constant': is_constant.values,
        'Unique values': unique_counts.values,
        'Value': [
            components_df[level].iloc[0] if is_constant[level] else None
            for level in TIMESTAMP_LEVELS
        ],
    })
    constant_summaries[column] = constant_summary
    display(constant_summary)

    constant_prefix = []
    for level in TIMESTAMP_LEVELS:
        if unique_counts[level] != 1:
            break
        constant_prefix.append(f'{level}={components_df[level].iloc[0]}')

    if constant_prefix:
        print(f"Constant across this timestamp column (coarsest to finest): {', '.join(constant_prefix)}")
    else:
        print("No timestamp element is constant across this timestamp column, not even the year.")

### 3. Detect timestamp format

In [ ]:

def infer_timestamp_format_from_column(series):
    """
    Infer a common timestamp format from a pandas Series.

    Supports:
      %Y-%m-%d
      %Y-%d-%m
      %m-%d-%Y
      %d-%m-%Y

    with optional:
      HH:MM:SS
      timezone offset like +00:00
    """

    candidates = {
        "YMD": "%Y-%m-%d",
        "YDM": "%Y-%d-%m",
        "MDY": "%m-%d-%Y",
        "DMY": "%d-%m-%Y",
    }

    possible = set(candidates.keys())
    detected_suffix = None

    for value in series.dropna():
        s = str(value).strip()

        # Separate date and optional time
        match = re.match(
            r"^(\d{1,4})[-/](\d{1,2})[-/](\d{1,4})"
            r"(?:[ T](\d{2}):(\d{2}):(\d{2})([+-]\d{2}:\d{2}|Z)?)?$",
            s
        )

        if not match:
            continue

        a, b, c, hour, minute, second, tz = match.groups()
        a, b, c = int(a), int(b), int(c)

        valid_for_row = set()

        # YYYY-MM-DD
        if len(match.group(1)) == 4 and 1 <= b <= 12 and 1 <= c <= 31:
            valid_for_row.add("YMD")

        # YYYY-DD-MM
        if len(match.group(1)) == 4 and 1 <= b <= 31 and 1 <= c <= 12:
            valid_for_row.add("YDM")

        # MM-DD-YYYY
        if len(match.group(3)) == 4 and 1 <= a <= 12 and 1 <= b <= 31:
            valid_for_row.add("MDY")

        # DD-MM-YYYY
        if len(match.group(3)) == 4 and 1 <= a <= 31 and 1 <= b <= 12:
            valid_for_row.add("DMY")

        possible &= valid_for_row

        # Infer time suffix
        if hour is not None:
            suffix = " %H:%M:%S"
            if "T" in s:
                suffix = "T%H:%M:%S"

            if tz is not None:
                suffix += "%z"

            detected_suffix = suffix

        if len(possible) == 1:
            break

    if len(possible) == 1:
        key = next(iter(possible))
        return candidates[key] + (detected_suffix or "")

    if len(possible) > 1:
        return {
            "status": "AMBIGUOUS",
            "possible_formats": [
                candidates[x] + (detected_suffix or "")
                for x in sorted(possible)
            ]
        }

    return None

In [ ]:
format_results = {}

for column in TIMESTAMP_COLUMNS:
    fmt = infer_timestamp_format_from_column(
        event_log[column]
    )
    format_results[column] = fmt
    print(f"{column}: {fmt}")

### 4. Detect timezone

In [ ]:
def identify_timezone(series):
    """Identify timezone information for a pandas datetime Series."""
    non_missing = series.dropna()

    if non_missing.empty:
        return {
            'Timezone-aware': None,
            'Timezone': None,
            'UTC offset(s)': None,
            'DST varies': None,
        }

    # For datetime64[ns, tz] columns, pandas stores one timezone for the column.
    tz = getattr(series.dt, 'tz', None)

    if tz is None:
        return {
            'Timezone-aware': False,
            'Timezone': None,
            'UTC offset(s)': None,
            'DST varies': None,
        }

    # A named timezone may have different UTC offsets over the observed period
    # because of daylight-saving time. Report all offsets actually observed.
    offsets = sorted({
        ts.utcoffset().total_seconds() / 3600
        for ts in non_missing
        if ts.utcoffset() is not None
    })

    def format_offset(hours):
        sign = '+' if hours >= 0 else '-'
        hours = abs(hours)
        h = int(hours)
        m = int(round((hours - h) * 60))
        return f"UTC{sign}{h:02d}:{m:02d}"

    formatted_offsets = ', '.join(format_offset(x) for x in offsets)

    return {
        'Timezone-aware': True,
        'Timezone': str(tz),
        'UTC offset(s)': formatted_offsets,
        'DST varies': len(offsets) > 1,
    }


timezone_rows = []
for column in TIMESTAMP_COLUMNS:
    info = identify_timezone(event_log[column])
    timezone_rows.append({'Timestamp column': column, **info})

timezone_summary = pd.DataFrame(timezone_rows)
display(timezone_summary)

for row in timezone_rows:
    column = row['Timestamp column']
    if row['Timezone-aware'] is True:
        print(f"{column}: timezone={row['Timezone']}, observed offset(s)={row['UTC offset(s)']}")
    elif row['Timezone-aware'] is False:
        print(f"{column}: timezone-naive — no timezone information is encoded in the timestamps.")
    else:
        print(f"{column}: no non-missing timestamps available.")


In [ ]:
from ipywidgets import interact, Dropdown
import warnings

In [ ]:
GRANULARITY_LEVELS = ['day', 'hour', 'minute', 'second', 'millisecond']
FREQ_ALIASES = {'day': 'D', 'hour': 'h', 'minute': 'min', 'second': 's', 'millisecond': 'ms'}

timestamp_widget = Dropdown(
    options=[('Select a timestamp column...', None)] + [(col, col) for col in TIMESTAMP_COLUMNS],
    value=None,
    description='Timestamp:',
)

granularity_widget = Dropdown(
    options=[('Select a granularity...', None)] + [(level, level) for level in GRANULARITY_LEVELS],
    value=None,
)

modified_event_log = event_log.copy()

@interact(timestamp_column=timestamp_widget, target_granularity=granularity_widget)
def renormalize_timestamps(timestamp_column, target_granularity):
    if timestamp_column is None or target_granularity is None:
        return

    freq = FREQ_ALIASES[target_granularity]
    original = modified_event_log[timestamp_column]
    rounded = original.dt.round(freq)

    # Period dtype at the target freq structurally cannot hold finer-than-target
    # precision (e.g. period[D] has no time component at all), unlike datetime64
    # which would silently keep sub-second precision around after rounding.
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)  # tz dropped on purpose: no longer meaningful below day/hour granularity
        renormalized = rounded.dt.to_period(freq)

    changed = (rounded != original).sum()
    new_column = f'{timestamp_column}:renormalized:{target_granularity}'

    modified_event_log[new_column] = renormalized

    print(
            f"{changed}/{len(original)} timestamps in '{timestamp_column}' "
            f'rounded to the nearest {target_granularity}'
        )
    print(f'Created column: {new_column}')
    print(f'Renormalized column dtype: {renormalized.dtype}')

In [ ]:
display(modified_event_log.head())